In [2]:
!pip install dash
!pip install dash-bootstrap-components
!pip install nltk
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.1 MB/s eta 0:00:00


In [3]:
import dash
from dash import dcc, html, dash_table, Input, Output, State, callback
import dash_bootstrap_components as dbc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
import base64
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm
import os

# Initialize Dash app
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.title = "OmanAI - Sentiment Analysis"
server = app.server

# Layout components
header = dbc.Navbar(
    dbc.Container([
        dbc.NavbarBrand("OmanAI Sentiment Analyzer", className="ms-2"),
        dbc.Nav([
            dbc.NavItem(dbc.NavLink("Home", href="#")),
            dbc.NavItem(dbc.NavLink("Contact", href="#")),
        ], className="ms-auto"),
    ]),
    color="primary",
    dark=True,
    sticky="top"
)

upload_section = dbc.Card([
    dbc.CardHeader("Upload Customer Feedback Data"),
    dbc.CardBody([
        dcc.Upload(
            id='upload-data',
            children=html.Div([
                'Drag and Drop or ',
                html.A('Select Excel File')
            ]),
            style={
                'width': '100%',
                'height': '60px',
                'lineHeight': '60px',
                'borderWidth': '1px',
                'borderStyle': 'dashed',
                'borderRadius': '5px',
                'textAlign': 'center',
                'margin': '10px'
            },
            multiple=False
        ),
        dbc.Alert(
            "File should contain a 'tweet' column with customer feedback text",
            color="info",
            className="mt-2"
        )
    ])
])

controls = dbc.Card([
    dbc.CardHeader("Analysis Controls"),
    dbc.CardBody([
        dbc.Button("Run Sentiment Analysis",
                  id="analyze-button",
                  color="primary",
                  className="w-100 mb-3"),
        dbc.Spinner(html.Div(id="loading-output"))
    ])
])

contact_info = dbc.Card([
    dbc.CardHeader("Contact Information"),
    dbc.CardBody([
        html.P("OmanAI Headquarters: Mansfield, TX USA"),
        html.P("Founder: Daniel D Dompreh (MBA)"),
        html.P("Email: domprehd@yahoo.com"),
        html.P("Phone: +1 (614) 218-4856"),
        html.P("Website: omanai.com"),
        html.Img(src="https://omanai.com/logo.png", height=50, className="mt-2")
    ])
])

# App layout
app.layout = dbc.Container([
    header,
    dbc.Row([
        dbc.Col(upload_section, width=8),
        dbc.Col([controls, contact_info], width=4)
    ], className="mt-4"),
    dbc.Row([
        dbc.Col(dcc.Graph(id='sentiment-chart'), width=6),
        dbc.Col(dcc.Graph(id='pie-chart'), width=6)
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(id='model-comparison-chart'), width=12)
    ]),
    dbc.Row([
        dbc.Col([
            html.H5("Analysis Results", className="mt-4"),
            html.Div(id='results-table'),
            html.Div([
                dbc.Button("Download Full Report",
                           id="download-button",
                           color="success",
                           className="me-2"),
                dcc.Download(id="download-excel"),
                dcc.Download(id="download-report")
            ], className="mt-3 mb-5")
        ])
    ])
], fluid=True)

# Initialize models (cached for performance)
@callback(
    Output('vader-model', 'data'),
    Input('app-ready', 'data')
)
def initialize_vader(_):
    nltk.download('vader_lexicon')
    return {'model': SentimentIntensityAnalyzer()}

@callback(
    Output('roberta-model', 'data'),
    Input('app-ready', 'data')
)
def initialize_roberta(_):
    MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL)
    return {'tokenizer': tokenizer, 'model': model}

# Main analysis callback
@callback(
    [Output('sentiment-chart', 'figure'),
     Output('pie-chart', 'figure'),
     Output('model-comparison-chart', 'figure'),
     Output('results-table', 'children'),
     Output('download-excel', 'data'),
     Output('download-report', 'data'),
     Output('loading-output', 'children')],
    [Input('analyze-button', 'n_clicks')],
    [State('upload-data', 'contents'),
     State('upload-data', 'filename'),
     State('vader-model', 'data'),
     State('roberta-model', 'data')],
    prevent_initial_call=True
)
def run_analysis(n_clicks, contents, filename, vader_data, roberta_data):
    if not contents or n_clicks is None:
        return dash.no_update

    # Parse uploaded file
    content_type, content_string = contents.split(',')
    decoded = base64.b64decode(content_string)

    try:
        df = pd.read_excel(BytesIO(decoded))
    except Exception as e:
        return [dash.no_update] * 7 + [dbc.Alert(f"Error reading file: {str(e)}", color="danger")]

    # Clean data
    df_cleaned = df.dropna(subset=['tweet']).copy()

    # Initialize models
    sia = vader_data['model']
    tokenizer = roberta_data['tokenizer']
    roberta_model = roberta_data['model']

    # VADER Analysis
    vader_results = []
    for text in tqdm(df_cleaned['tweet'], desc="VADER Analysis"):
        vader_results.append(sia.polarity_scores(str(text)))
    vader_df = pd.DataFrame(vader_results)
    vader_results_df = pd.concat([df_cleaned.reset_index(drop=True), vader_df], axis=1)

    # RoBERTa Analysis
    def roberta_sentiment(text):
        encoded_text = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        output = roberta_model(**encoded_text)
        scores = softmax(output[0][0].detach().numpy())
        return {'roberta_neg': scores[0], 'roberta_neu': scores[1], 'roberta_pos': scores[2]}

    roberta_results = []
    for text in tqdm(df_cleaned['tweet'], desc="RoBERTa Analysis"):
        roberta_results.append(roberta_sentiment(str(text)))
    roberta_df = pd.DataFrame(roberta_results)
    full_results = pd.concat([vader_results_df, roberta_df], axis=1)

    # Generate visualizations
    ## Sentiment distribution
    sentiment_counts = {
        'Positive': (full_results['compound'] > 0).sum(),
        'Neutral': (full_results['compound'] == 0).sum(),
        'Negative': (full_results['compound'] < 0).sum()
    }

    bar_fig = {
        'data': [{
            'x': list(sentiment_counts.keys()),
            'y': list(sentiment_counts.values()),
            'type': 'bar',
            'marker': {'color': ['green', 'blue', 'red']}
        }],
        'layout': {
            'title': 'Sentiment Distribution',
            'xaxis': {'title': 'Sentiment'},
            'yaxis': {'title': 'Count'}
        }
    }

    pie_fig = {
        'data': [{
            'labels': list(sentiment_counts.keys()),
            'values': list(sentiment_counts.values()),
            'type': 'pie',
            'marker': {'colors': ['green', 'blue', 'red']},
            'hoverinfo': 'label+percent',
            'textinfo': 'value'
        }],
        'layout': {'title': 'Sentiment Proportions'}
    }

    ## Model comparison
    model_comparison = pd.DataFrame({
        'VADER': [
            full_results['neg'].mean(),
            full_results['neu'].mean(),
            full_results['pos'].mean()
        ],
        'RoBERTa': [
            full_results['roberta_neg'].mean(),
            full_results['roberta_neu'].mean(),
            full_results['roberta_pos'].mean()
        ]
    }, index=['Negative', 'Neutral', 'Positive'])

    comparison_fig = {
        'data': [
            {'x': model_comparison.index,
             'y': model_comparison['VADER'],
             'type': 'bar',
             'name': 'VADER',
             'marker': {'color': 'blue'}},
            {'x': model_comparison.index,
             'y': model_comparison['RoBERTa'],
             'type': 'bar',
             'name': 'RoBERTa',
             'marker': {'color': 'orange'}}
        ],
        'layout': {
            'title': 'Model Comparison: Sentiment Scores',
            'barmode': 'group',
            'xaxis': {'title': 'Sentiment'},
            'yaxis': {'title': 'Average Score'}
        }
    }

    # Create results table
    results_table = dash_table.DataTable(
        columns=[{"name": i, "id": i} for i in full_results.columns],
        data=full_results.head(10).to_dict('records'),
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={
            'height': 'auto',
            'minWidth': '100px', 'width': '150px', 'maxWidth': '300px',
            'whiteSpace': 'normal'
        }
    )

    # Prepare download files
    excel_buffer = BytesIO()
    full_results.to_excel(excel_buffer, index=False)
    excel_buffer.seek(0)

    # Create PDF report (simplified)
    report_content = f"""
    OmanAI Sentiment Analysis Report

    Analysis Summary:
    - Total Feedback Items: {len(full_results)}
    - Positive Sentiment: {sentiment_counts['Positive']} ({sentiment_counts['Positive']/len(full_results):.1%})
    - Neutral Sentiment: {sentiment_counts['Neutral']} ({sentiment_counts['Neutral']/len(full_results):.1%})
    - Negative Sentiment: {sentiment_counts['Negative']} ({sentiment_counts['Negative']/len(full_results):.1%})

    Recommendations:
    1. Address top 3 negative feedback themes
    2. Enhance features with positive sentiment
    3. Monitor neutral feedback for improvement opportunities

    OmanAI - Transforming Customer Insights
    Mansfield, TX USA | domprehd@yahoo.com | +1 (614) 218-4856
    """

    report_buffer = BytesIO()
    report_buffer.write(report_content.encode())
    report_buffer.seek(0)

    return (
        bar_fig,
        pie_fig,
        comparison_fig,
        results_table,
        dict(content=excel_buffer.read(), filename="sentiment_results.xlsx"),
        dict(content=report_buffer.read(), filename="analysis_report.txt"),
        ""
    )

# Hidden stores for models
app.layout.children.append(html.Div([
    dcc.Store(id='vader-model'),
    dcc.Store(id='roberta-model'),
    dcc.Store(id='app-ready', data=True)  # Triggers model initialization
]))

if __name__ == '__main__':
    app.run(port=8051)

<IPython.core.display.Javascript object>